# 1.1 CAD Basics

We consider a concrete example to show our approach to Cylindrical Algebraic
Decomposition. Recall that in general, CAD consists of a recursive projection
phase that projects equations to one dimension lower, until we reach dimension
\1. We follow this with a recursive lifting phase, which starts at the projected
equations of dimension 1 and lifts sample points to the highest dimension, one
dimension at a time. The result is a list of sample points for each
sign-invariant cell of the CAD.

## Our Example

Our implementation is motivated by neural network verification. A central
problem here is the ReLU function. Consider the following ReLU-inspired formula:

$$

f(x) = 
    \begin{cases}
        2x & \text{if } x > 0\\
        0 & \text{if } x \leq 0
    \end{cases}
$$

Now say we want to verify the following formula, where we add $F$ to our first
order logic vocabulary to represent the function evaluation:

$\exists x . F(x) > 5 \land x < 5$

To "compile" $F(x)$, we introduce a new variable $u$ representing the function
output:

$F(x) = u \land u > 5$

Then we "plug in" our ReLU-like function, making the entire formula:

$$
\exists x \exists u . 
((x < 0 \land u = 0) \lor (x \geq 0 \land u = 2x)) 
\land (u > 5) 
\land (x < 5)
$$

Now to be more general, take $x_1$ to be $x$ and $x_2$ to be $u$. Our formula
contains the following constraints:

- $x_1 < 0$
- $x_1 \geq 0$
- $x_1 - 5 < 0$
- $x_2 = 0$
- $-2x_1 + x_2 = 0$
- $-5 + x_2 = 0$

This will be the input to our projection phase. First, we project $x_2$. To do
this, we take all formulas of $x_2$'s dimension:

- $x_2 = 0$
- $-2x_1 + x_2 = 0$
- $-5 + x_2 = 0$

Of these, we calculate the pairwise intersections by moving $x_2$ to the
right-hand side of each equation and equating them:

- $0 = -2 $x_1$ -> $x_1 = 0$
- $-5 = 0$ -> no solution
- $-2x_1 = -5$ -> $x_1 = 2.5$

Now we also add all equations that only contain $x_1$; the "vertical"
constraints. This is $x_1 = 0$, which we already have, and $x_1 - 5 < 0$, so
$x_1 = 5$.

Thus, our resulting points in dimension 1 are 0, 2.5, and 5. These will form our
intervals:

- $(\infty, 0)$
- $(0,0)$
- $(0, 2.5)$
- $(2.5, 2.5)$
- $(2.5, 5)$
- $(5, 5)$
- $(5, +\infty)$

We lift these up by taking a sample of each interval.

TODO: continue here

Dit kunnen we "omhoog" liften door telkens een sample punt te nemen en dit in te
vullen in de bovenliggende formules. Concreet geeft dit de intervallen (2.5,
2.5) en (2.5, 5) die een SAT opleveren.

---



Wat als we een stap moeilijker gaan. Neem $f(x,y) = max(0, x + y)$ en query
$\exists x \exists y . F(x,y) > 2 \land x < 2 \land y < 2$

De volledige formule is dan:

$$
\exists x,y,z .
((x + y < 0 \land z=0) \lor (x + y \geq 0 \land z = x + y))
\land (z > 2) \land (x < 2) \land (y < 2)
$$

Alle constraints zijn dan:

1. $x + y$
2. $z$
3. $x + y - z$
4. $z - 2$
5. $x - 2$
6. $y - 2$

**Stap 1: wegprojecteren z**. Alle formules waar z in meespeelt zijn 2, 3, en 4.
Bereken de snijpunten:

- 2x3: $x + y = 0$
- 2x3: geen oplossing
- 3x4: $x + y -2 = 0$

Dan voegen we ook de andere constraints weer toe:

1. $x + y$ (2x3 maar ook 1)
2. $x + y - 2$
3. $x - 2$
4. $y - 2$

**Stap 2: wegprojecteren y**. Alle formules waar y in meespeelt zijn 1, 2, en 4.

- 1x2: geen oplossing
- 1x4: $x + 2= 0$
- 2x4: $x = 0$

En we voegen $x = 2$ weer toe.

**Stap 3: lifting**. De vorige stap geeft ons volgende intervallen:

- $(-\infty, -2)$
- $(-2, -2)$
- $(-2, 0)$
- $(0, 0)$
- $(0, 2)$
- $(2, 2)$
- $(2, +\infty)$

Deze moeten we één voor één afgaan en naar boven liften.

- $(-\infty, -2)$
  - Neem sample point $x=-4$
  - Dan (constraints stap 1): $y=2$, $y=4$, $y=6$. Sample hier steeds een punt
    uit:
    - $(-4, 0)$ -> UNSAT
    - $(-4, 2)$ -> UNSAT
    - $(-4, 3)$ -> UNSAT
    - $(-4, 4)$ -> UNSAT
    - $(-4, 5)$ -> UNSAT
    - $(-4, 6)$ -> UNSAT
    - $(-4, 7)$ -> UNSAT
- $(-2, -2)$
  - Neem "sample point" $x=-2$
  - Dan: $y=2$, $y=4$
    - $(-2, 0)$ -> UNSAT
    - Rest ook UNSAT, want $y<2$ altijd false.
- etc. etc, e.g.
- $(0, 2)$
  - Neem "sample point" $x=1$
  - Dan (constraints stap 1): $y=-1$, $y=1$, $y=2$. Sample hier steeds een punt uit:
    - $(1,-2)$ -> z<2, dus UNSAT
    - $(1, -1)$ -> UNSAT
    - $(1, 0)$ -> UNSAT
    - $(1, 1)$ -> UNSAT
    - $(1, 1.5)$ -> SAT
    - $(1, 2)$ -> UNSAT
    - $(1, 3)$ -> UNSAT
  - Hier vinden we dus een `SAT` en mogen we stoppen.

In [1]:
import duckdb as db
import numpy as np
import pandas as pd
from IPython.display import display
import plotly.graph_objects as go
import plotly.express as px
from dataclasses import dataclass

In [2]:

con = db.connect("cad.db")

def create_db_with_constraints(con, constraints):
    con.sql("DROP TABLE IF EXISTS LinearConstraint")
    con.sql("""
        CREATE TABLE LinearConstraint(
            id INTEGER PRIMARY KEY,
            description VARCHAR NOT NULL
        )
    """)

    con.sql("DROP TABLE IF EXISTS Coefficient")
    con.sql("""
        CREATE TABLE Coefficient(
            constraint_id VARCHAR NOT NULL, -- VARCHAR omdat we dingen zoals 1x2 willen doen
            dimension INTEGER NOT NULL,
            value DOUBLE NOT NULL,
            PRIMARY KEY (constraint_id, dimension)
        )
    """)


    for constraint_id, coeffs in enumerate(constraints):
        con.execute("""
            INSERT INTO LinearConstraint(id, description)
            VALUES ($id, $description)
        """, {'id': constraint_id, 'description': coeffs[0]})

        for dimension, coeff_value in enumerate(coeffs[1:]):
            con.execute("""
                INSERT INTO Coefficient(constraint_id, dimension, value)
                VALUES ($constraint_id, $dimension, $value)
            """, {'constraint_id': constraint_id, 'dimension': dimension, 'value': coeff_value})

    con.execute("EXPORT DATABASE 'cadinsql.db'")

constraints = [
    ["x+y", 0, 1, 1],
    ["z", 0, 0, 0, 1],
    ["x+y-z", 0, 1, 1, -1],
    ["z-2", -2, 0, 0, 1],
    ["x-2", -2, 1],
    ["y-2", -2, 0, 1]
]

create_db_with_constraints(con, constraints)

Ik heb eerst een niet-recursieve query geschreven. Die kunnen we uitvoeren om
dezelfde oplossing te bekomen als die we manueel hadden gevonden:

In [3]:
with open('queries/cad_qe_nonrecursive.sql') as file:
    query = file.read()

con.sql(query)

FileNotFoundError: [Errno 2] No such file or directory: 'queries/cad_qe_nonrecursive.sql'

Paar notities.

**Berekenen intersects**. We zitten met lineaire constraints. Om de intersect
tussen 2 te berekenen naar z toe (wegprojecteren z): 

- Normaliseer naar z=1 (deel andere coeffs door z)
- Trek de coeffs van de 2 vergelijkingen paarsgewijs van elkaar af
- Resultaat: intersect vergelijking
- Als alles 0: geen intersect

**SAT checking**. Als laatste zetten we onze query om in SQL, om te kijken of er
inputs zijn die SAT geven. Kan dit beter/generieker? 

---

Recursieve versie (zelfde resultaat)

In [ ]:
with open('queries/cad_qe_recursive.sql') as file:
    query = file.read()

con.sql(query)

┌──────────┬───────────┬────────┐
│ input_id │ dimension │ value  │
│ varchar  │   int32   │ double │
├──────────┼───────────┼────────┤
│ 5.4.2    │         1 │    1.0 │
│ 5.4.2    │         2 │    1.5 │
│ 5.4.2    │         3 │    2.5 │
└──────────┴───────────┴────────┘

---

Nu kunnen we dus een query bedenken van een dimensie meer en die proberen uit te voeren.

Neem $f(x,y, z) = max(0, x + y - z)$ en query
$\exists x,y,z . F(x,y,z) > 2 \land x < 4 \land y < 4 \land z > 2$


In [ ]:

constraints = [
    ["x+y-z", 0, 1, 1, -1],
    ["u", 0, 0, 0, 0, 1],
    ["x+y-z-u", 0, 1, 1, -1, -1],
    ["u-2", -2, 0, 0, 0, 1],
    ["x-4", -4, 1],
    ["y-4", -4, 0, 1],
    ["z-2", -2, 0, 0, 1]
]

create_db_with_constraints(con, constraints)

with open('queries/cad_qe_recursive_example2.sql') as file:
    query = file.read()

con.sql(query)

┌──────────┬───────────┬────────┐
│ input_id │ dimension │ value  │
│ varchar  │   int32   │ double │
├──────────┼───────────┼────────┤
│ 5.6.2.2  │         1 │    1.0 │
│ 5.6.2.2  │         2 │    3.5 │
│ 5.6.2.2  │         3 │   2.25 │
│ 5.6.2.2  │         4 │   2.25 │
│ 6.6.2.2  │         1 │    2.0 │
│ 6.6.2.2  │         2 │    2.5 │
│ 6.6.2.2  │         3 │   2.25 │
│ 6.6.2.2  │         4 │   2.25 │
│ 7.6.2.2  │         1 │    3.0 │
│ 7.6.2.2  │         2 │    1.5 │
│ 7.6.2.2  │         3 │   2.25 │
│ 7.6.2.2  │         4 │   2.25 │
├──────────┴───────────┴────────┤
│ 12 rows             3 columns │
└───────────────────────────────┘

Opties om verder te bekijken:

- Fourier-Motzkin Elimination
- Ferrante-Rackoff
- Linear programming

In [ ]:
con.close()